# 🫀 퀘스트 46 · Q9-G1′ — **자를 고치고 리듬 기저를 팔로** (G1 재설계)

| | **MedKOS / `notebooks/quest46_q9_g1p_pmorph_v2.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` — 층① 표현 |
| 부모 런 | `quest46_q8_g1_personal_pmorph` (공식 실행 `20260804T1340`) |
| 성격 | **재설계 런** — 1차는 판정이 아니라 **측정 실패**였다(R35 ①) |

## 1차가 왜 판정이 아니었나

```
raw          0.6475 · 영점 0.4921 · 셔플 0.6289      ← 셔플해도 안 떨어진다
cancel       0.6260 · 영점 0.4967 · 셔플 0.6285      ← 셔플이 **더 높다**
cancel_pmask 0.5858 · 영점 0.4924 · 셔플 0.6057      ← P 지워도 영점 훨씬 위
코호트 35명인데 통계량은 **n=11**
```

**넷이 겹쳐 있었다.**

| # | 문제 | 실측 근거 |
|---|---|---|
| ① | **코호트 세기가 틀린 걸 셌다** — S/N 개수를 셌는데 통계량이 요구하는 건 **매칭 쌍** | 35명 → **n=11** |
| ② | **매칭 층이 2.78ms** (`np.round(f1)` · `pre_rr` 단위가 샘플). S 는 정의상 f1 이 크고 N 은 0 근처라 **거의 안 겹친다** | S200×N1000 = 20만 쌍 중 **68쌍**(층 1샘플). 10샘플 675 · 20샘플 1,464 |
| ③ | **리듬 통제가 선형인데 모델은 비선형** — CNN 이 파형에서 심박 대리변수를 만들 수 있다(R24) | 셔플해도 성능 불변 = **정렬 비의존** |
| ④ | **P 마스크가 너무 좁다** — P 폭 80~110ms 인데 ±25ms 만 지웠다 | `pmask` 0.5858 에 **잔여 P** 가 섞였다 |

★ ②는 **Q7-AA 의 천장 0.5608/0.6097 에도 그대로 걸린다**(같은 코드·같은 층 · 56→34).
따라서 **앵커 0.6097 을 그대로 쓸 수 없다** — 자가 바뀌면 눈금도 다시 그어야 한다.

## 무엇을 바꾸나

1. **층 폭을 사전등록 상수로 뺀다**(`STRATUM_W`). 폭 민감도는 격자로 **보고**하되
   **판정은 사전등록 폭 하나**로 한다(문턱 훑기 금지 · R34 ②).
2. **코호트를 「매칭 쌍 수」로 센다** — 통계량이 실제로 요구하는 것을 센다(R11-b 의 원래 취지).
3. **천장을 런 안에서 다시 낸다** — 같은 환자·같은 구간·같은 층 폭의 `p_score`.
   외부 앵커 0.6097 은 **비교 불가**로 명시하고 병기만 한다.
4. ★★★ **리듬-only 팔을 넣는다**(= 사전등록 G3 를 이 런에 흡수). 별도 런으로 하면 코호트·
   분할·영점이 달라져 비교가 흔들린다. **같은 런의 짝지은 차**여야 「P 의 순수 몫」이 나온다.
5. **음성 대조를 둘로** — P 전폭 마스킹(`pmask`)과 **P 창 밖 다른 창**(`offwin`).

```
팔 = { rhythm , raw , cancel , pmask , offwin }   × 환자 안 시간 분할
```

⚠️ **`rhythm` 팔도 같은 잔차화·매칭을 거친다.** 그러면 선형 리듬은 지워지고 **비선형 리듬**만
남는데, 그게 바로 ③이 의심한 교란이다 — 즉 `rhythm` 팔이 **그 교란의 크기를 직접 잰다**.

## 관문 (재사전등록)

| 관문 | 무엇 | 통과 기준 |
|---|---|---|
| **H0** | 자산 항등 + **매칭 쌍 기준** 코호트 세기 | 정합·생리 타당성 · 코호트 미달이면 **중단** |
| **H1 ★ 자 교정** | 층 폭 민감도 + **런 내 천장** 재계산 | 관문 아님(자 세우기). 판정 폭은 사전등록 하나 |
| **H2 ★★★ 주 관문** | **`raw − rhythm` 짝지은 차** + `raw` vs 런 내 천장 | 차의 CI 가 0 을 뗄 것 **그리고** `raw` 가 천장 CI 상단 초과. 하나만이면 **미결** |
| **H3** | 파이프라인 영점(학습 라벨 치환 + 재학습) | 0.5 를 **가정하지 말고 측정** |
| **H4** | 정렬·위치 대조 — 앵커 셔플 · `p_idx` 단독 · ★ **`offwin`** | `offwin` ≈ `raw` 면 **P 창이 특별하지 않다** |
| **H5** | 결론 검산표 | R38 ⑦ · R39 ⑤ |

### 판정표

- **H2 ✅ · H4 ✅** → 층① 열림. 다음은 **G2 라벨-이득 곡선**(임상 결정 숫자는 AUROC 가 아니라 **N**)
- **H4 ❌**(`offwin` ≈ `raw`) → **어느 쪽이든 P 창이 특별하지 않다** → H2 를 P 의 증거로 읽지 않는다
- **H2 ❌/미결** → **층① 종결.** Q7-AA 의 해리가 「**환자 안에서도 리듬 너머로는 아니다**」로 좁혀진다

⚠️ **1차의 수(0.6475 등)를 인용하지 않는다** — 자가 달랐다. 이 런이 그 자리를 대체한다.
⚠️ **새 데이터 0** — `svdb_data5.npz` + `svdb_pdelin.npz`.


In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def mde(lo, hi):
    return (hi - lo) / 2.0 if np.isfinite(lo) and np.isfinite(hi) else float("nan")

def boot_mean(v, seed, nb=3000, q=2.5):
    d = np.asarray(v, float); d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), len(d)
    rng = np.random.RandomState(seed)
    b = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)]
    return (float(d.mean()), float(np.percentile(b, q)),
            float(np.percentile(b, 100 - q)), len(d))

def boot_pair(a, b, seed, nb=3000, q=2.5):
    """★ **짝지은 차**(b − a) — 같은 환자에서 두 팔을 재므로 짝을 유지한다."""
    a = np.asarray(a, float); b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b); a, b = a[m], b[m]
    if len(a) < 3:
        return float("nan"), float("nan"), float("nan"), len(a)
    rng = np.random.RandomState(seed)
    d = [(b[j] - a[j]).mean() for j in (rng.randint(0, len(a), len(a)) for _ in range(nb))]
    return (float((b - a).mean()), float(np.percentile(d, q)),
            float(np.percentile(d, 100 - q)), len(a))

def _rank_avg(v):
    v = np.asarray(v, float); o = v.argsort()
    r = np.empty(len(v), float); r[o] = np.arange(len(v), dtype=float)
    for u in np.unique(v):
        m = v == u
        if m.sum() > 1:
            r[m] = r[m].mean()
    return r

def spearman(a, b):
    """★★ 동점을 **평균 순위**로 처리한다(Q3-B 에서 argsort 판본의 순서 의존이 드러났다)."""
    ra, rb = _rank_avg(a), _rank_avg(b)
    if np.std(ra) < 1e-12 or np.std(rb) < 1e-12:
        return float("nan")
    return float(np.corrcoef(ra, rb)[0, 1])

def need_super(n, half, eff, p80=False):
    if not np.isfinite(half) or abs(eff) < 1e-9 or n < 1:
        return float("nan")
    r = float(n) * (half / abs(eff)) ** 2
    return r * 2.04 if p80 else r

class AssetError(RuntimeError): pass
print("CELL 0 ✅")


In [ ]:
# CELL 1 — 설정 · 재사전등록
import os, sys, json, importlib, time, warnings
importlib.invalidate_caches(); warnings.filterwarnings("ignore")

SMOKE = os.environ.get("MEDKOS_SMOKE") == "1"
_ENV_ROOT = os.environ.get("MEDKOS_DRIVE_ROOT")
if _ENV_ROOT:
    DRIVE_ROOT = _ENV_ROOT
else:
    try:
        from google.colab import drive; drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = "/content/drive/MyDrive"
    except Exception as e:
        print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0, IDX_S = 20260804, 1
FS, RPRE, L = 360, 100, 300
FULL_K = tuple(range(4, 33))
RHY_K  = (5, 10, 20, 32)

# ── ★★★ 사전등록 상수 (SMOKE 가 절대 안 건드린다)
STRATUM_W = 10           # ★★ 매칭 층 폭(샘플) = 27.8ms. 1차는 1샘플(2.78ms)이라 쌍이 죽었다
FRAC_TRAIN = 0.50
GUARD_S = 60.0
MIN_S_TR, MIN_N_TR = 25, 25    # 학습이 가능하려면 필요한 최소 라벨
MIN_PAIR = 200                 # ★ 코호트 조건이 **이것**이다(1차는 S/N 개수로 셌다)
MIN_REC = 8
HW_P = 32                      # P 창 반폭 → 65샘플 ≈ 181ms
PMASK_MS = 55.0                # ★ P 전폭(80~110ms)을 덮는다. 1차는 25ms 라 잔여 P 가 남았다
OFFWIN_C = 160                 # ★ P 창 밖 대조창 중심(R+60샘플 ≈ TP 구간)
ANCHOR_OLD = 0.6097            # ⚠️ 1차 앵커 — **층 폭이 달라 비교 불가**. 병기만 한다
W_GRID = (1, 2, 5, 10, 20, 40) # 층 폭 민감도(보고용 · 판정은 STRATUM_W 하나)

# ── 비용 손잡이(스모크에서만 축소)
NB_BOOT   = 400 if SMOKE else 3000
N_PERM_H3 = 1   if SMOKE else 3
N_SHUF_H4 = 1   if SMOKE else 2
EPOCHS = 200                   # ★ 설계 상수 — 풀배치라 EPOCHS = 그래디언트 스텝 수

ARMS = ("rhythm", "raw", "cancel", "pmask", "offwin")
WAVE_ARMS = ("raw", "cancel", "pmask", "offwin")     # 파형 창을 보는 팔
READ_ORDER = ("H0", "H1", "H2", "H3", "H4", "H5")
GATE_DEP = {"H1": ["H0"], "H2": ["H0", "H1"], "H3": ["H0"], "H4": ["H0", "H2"]}

SV5  = os.path.join(MITBIH, "svdb_data5.npz")
PDEL = os.path.join(MITBIH, "svdb_pdelin.npz")

REF = dict(
    g1_raw=0.6475, g1_cancel=0.6260, g1_pmask=0.5858, g1_n=11, g1_cohort=35,
    g1_null_raw=0.4921, g1_shuf_raw=0.6289, g1_base=0.5901,
    aa1=0.5608, aa1_hi=0.6097, aa5=-0.0007)

RULE_CHECK = {
    "R11-b 세기":       "★★★ 코호트를 **매칭 쌍 수**로 센다 — 1차는 S/N 개수를 세서 35→11 로 죽었다",
    "R16 fallback 없음": "자산 없으면 **중단**",
    "R22 선택 없음":     "고정 구조·고정 EPOCHS · 용량 격자 없음",
    "R24 / R27 기저":    "★★★ **리듬-only 팔**을 같은 런에 둔다 — 없으면 P 의 몫을 못 가른다",
    "R26 / R38 ②":      "영점은 **측정**한다",
    "R29 ② 분기 금지":   "H0 이 깨지면 아래를 **안 읽는다**",
    "R33 ① MDE":        "관문마다 MDE 를 내고 점추정과 비교. **미결 ≠ 등가**",
    "R34 ② 문턱 단발":   "★★ 층 폭 민감도는 **보고**하되 판정은 사전등록 폭 **하나**로",
    "R35 ① 자 먼저":     "★★★ 층 폭이 바뀌었으므로 **천장을 런 안에서 다시 낸다**",
    "R36 ⑤ 성분 병기":   "차는 **성분과 함께만** 인용",
    "R38 ⑦ 요약 정합":   "요약·검산표·판정이 **같은 갈래**여야 한다",
    "누출 차단":        "시간 분할 + 가드밴드 · 소거 템플릿·표준화 모두 **학습 구간에서만**",
}

CONFIG = dict(
    exp="quest46_q9_g1p_pmorph_v2", quest="ailab-2026-0046",
    step="personalized-p-morph-g1p",
    parent_exp=["quest46_q8_g1_personal_pmorph", "quest46_q7aa_burden_target"],
    scope="H0~H5. G2(라벨-이득)·G6(부담 층화)는 H2 통과 후",
    purpose=("**재설계 런 — 1차는 판정이 아니라 측정 실패였다.** 넷이 겹쳤다: "
             "①코호트를 S/N 개수로 세서 35명 중 **11명**만 통계량이 섰다(요구는 매칭 쌍) · "
             "②매칭 층이 **2.78ms**(`pre_rr` 단위가 샘플인데 `np.round(f1)`)라 S 와 N 이 "
             "거의 안 겹쳤다(S200×N1000 = 20만 쌍 중 **68쌍**) · ③리듬 통제가 **선형**인데 "
             "CNN 은 **비선형 심박 대리변수**를 만들 수 있다(R24) — 셔플해도 성능이 안 떨어진 게 "
             "그 증상이다 · ④P 마스크가 ±25ms 라 **잔여 P** 가 남았다(P 폭 80~110ms). "
             "★★★ 그래서 자를 고치고(층 폭·쌍 기준 코호트·런 내 천장) **리듬-only 팔을 넣는다**"
             "(사전등록 G3 를 이 런에 흡수 — 별도 런이면 코호트·영점이 달라져 비교가 흔들린다). "
             "주 관문은 **`raw − rhythm` 짝지은 차** 이고 그게 **P 의 순수 몫**이다."),
    dataset="SVDB — svdb_data5.npz + svdb_pdelin.npz (새 데이터 0)",
    arms=list(ARMS), read_order=READ_ORDER, gate_dep=GATE_DEP,
    stratum_w=STRATUM_W, stratum_ms=STRATUM_W / FS * 1000.0, w_grid=list(W_GRID),
    min_pair=MIN_PAIR, pmask_ms=PMASK_MS, offwin_c=OFFWIN_C, hw_p=HW_P,
    anchor_old=ANCHOR_OLD, epochs=EPOCHS,
    n_boot=NB_BOOT, n_perm_h3=N_PERM_H3, n_shuf_h4=N_SHUF_H4, smoke=SMOKE,
    ref=REF, rule_check=RULE_CHECK,
    predictions={
        "H0": f"자산 항등 + **매칭 쌍 기준** 코호트. 평가 구간 매칭 쌍 ≥ {MIN_PAIR} 이고 "
              f"학습 구간 S≥{MIN_S_TR} & N≥{MIN_N_TR} 인 환자. {MIN_REC}명 미만이면 **중단**",
        "H1": f"★ **자 세우기(관문 아님).** 층 폭 {W_GRID} 에서 쌍 수와 천장을 함께 찍고, "
              f"**판정은 사전등록 폭 {STRATUM_W}샘플({STRATUM_W/FS*1000:.1f}ms) 하나**로 한다. "
              "런 내 천장 = 같은 환자·같은 구간·같은 폭의 `p_score`. "
              f"⚠️ 1차 앵커 {ANCHOR_OLD} 는 **층 폭이 달라 비교 불가**이므로 병기만 한다",
        "H2": "★★★ **주 관문 — `raw − rhythm` 짝지은 차.** 두 팔이 **같은 환자·같은 구간·같은 "
              "통제**를 거치므로 차이가 곧 **P 창이 리듬 너머로 더하는 몫**이다. "
              "통과 = 차의 CI 가 0 을 뗄 것 **그리고** `raw` 가 런 내 천장 CI 상단을 넘을 것. "
              "**하나만이면 미결.** ⚠️ `rhythm` 팔도 같은 잔차화를 거치므로 거기 남는 건 "
              "**비선형 리듬**이고, 그게 1차에서 의심한 교란의 크기다",
        "H3": f"**파이프라인 영점** — 학습 구간 라벨을 환자 안에서 치환하고 재학습(reps={N_PERM_H3})",
        "H4": "**정렬·위치 대조** — ⓐ 앵커 셔플 ⓑ `p_idx` 단독(학습 없음) ⓒ ★ **`offwin`**"
              f"(중심 {OFFWIN_C} · P 창 밖). ★★ `offwin` ≈ `raw` 면 **P 창이 특별하지 않다**는 "
              "뜻이고, 그러면 H2 를 P 의 증거로 읽지 않는다",
        "H5": "결론 검산표"},
    caveat=("★★★ **1차의 수(raw 0.6475 등)를 인용하지 않는다** — 자가 달랐다. "
            "★★ `rhythm` 팔은 입력 차원이 9 라 은닉 48 로 충분하다 — 용량 부족으로 지는 게 "
            "아니다. ★ 소거 `abs` 는 적합이 없어 P 창까지 차감되므로 「그 환자 중앙 비트 대비 "
            "편차」를 잰다. ★ **문턱 훑기 금지** — 층 폭 격자는 보고용이고 판정은 하나다(R34 ②). "
            "★ **반대 증거**: Q7-S′ S1≈S2 · Q7-Z(자를 고쳐도 안 올랐다) · 1차의 정렬 비의존성."))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q9_g1p_pmorph_v2", CONFIG, project=PROJECT)
run.log("설정 ✅ **Q9-G1′ — 자를 고치고 리듬 기저를 팔로**")
run.log("  ★★★ 주 관문 = **`raw − rhythm` 짝지은 차** (P 의 순수 몫)")
run.log(f"  ★★ 층 폭 **{STRATUM_W}샘플({STRATUM_W/FS*1000:.1f}ms)** — 1차는 1샘플(2.8ms)이라 쌍이 죽었다")
run.log(f"  ★★ 코호트를 **매칭 쌍 수**로 센다(≥{MIN_PAIR}) — 1차는 S/N 개수로 세서 35→11")
run.log(f"  ★ 팔 {len(ARMS)}개 — {' · '.join(ARMS)}")
run.log(f"  ⚠️ 1차 앵커 {ANCHOR_OLD} 는 **비교 불가**(층 폭이 다르다) — 런 내 천장으로 대체한다")
if SMOKE:
    run.log(f"  ⚠️ **스모크런** — 비용 손잡이만 축소(H3 reps={N_PERM_H3} · H4 reps={N_SHUF_H4} · "
            f"NB_BOOT={NB_BOOT}). 관문 문턱·EPOCHS 는 그대로다")
run.log("\n  재사전등록 규칙 체크리스트 (R29 ③)")
for k_, v_ in RULE_CHECK.items():
    run.log(f"    [x] {k_:<18} {v_}")


In [ ]:
# CELL 2 — 【H-0】 자산 · 시간 분할 · ★★ 매칭 쌍 기준 코호트
import pandas as pd
run.log("\n" + "=" * 100)
run.log("【H-0】 자산 항등 · 시간 분할 · **매칭 쌍 기준** 코호트")
run.log("=" * 100)
VERD, DIFF = {}, {}
def g_(k, v, d):
    VERD[k] = v; run.log(f"  {k:<5}{v}  {d}")

for p_, why in ((SV5, "svdb_labels.py build"), (PDEL, "Q7-P0")):
    if not os.path.exists(p_):
        raise AssetError(f"{p_} 없음 — {why}(R16)")
D5 = np.load(SV5, allow_pickle=True); PD = np.load(PDEL, allow_pickle=True)
PID = np.asarray(D5["pid"]).astype(int); SYM = np.asarray(D5["sym"]).astype(str)
Y3 = np.asarray(D5["y3"]).astype(int)
PRE = np.asarray(D5["pre_rr"], float); POST = np.asarray(D5["post_rr"], float)
pid2 = np.asarray(PD["pid"]).astype(int); sym2 = np.asarray(PD["sym"]).astype(str)
if len(pid2) != len(PID):
    raise AssetError(f"길이 불일치 — d5 {len(PID)} vs pdelin {len(pid2)}")
bad = np.where((pid2 != PID) | (sym2 != SYM))[0]
if len(bad):
    raise AssetError(f"정합 깨짐 — 첫 불일치 idx {int(bad[0])}")
run.log(f"  자산 정합 ✅ (pid, sym) 원소 단위 일치 — {len(PID):,} 비트")

P_IDX = np.asarray(PD["p_idx"]).astype(int)
P_SC = np.asarray(PD["p_score"], float)
R_SMP = np.asarray(PD["r_samp"], float)
K = np.where(Y3 >= 0)[0]
RID = PID[K]; Y = Y3[K]; TT = (Y == IDX_S)
pre = PRE[K].astype(float); post = POST[K].astype(float)
pidx0 = P_IDX[K].copy(); psc_asset = P_SC[K].copy(); rsmp = R_SMP[K].copy()
XB = np.ascontiguousarray(np.asarray(D5["beat"])[K][:, 0, :]).astype(float)
RS = np.array(sorted(set(RID.tolist())))

FIRE = pidx0 >= 0
pr_med = float(np.median((RPRE - pidx0[FIRE]) / FS * 1000.0))
run.log(f"  P 검출 {FIRE.mean():.3f} · PR 중앙 **{pr_med:.1f}ms** (타당 100~230)")
if not (100.0 <= pr_med <= 230.0):
    raise AssetError(f"P 좌표가 생리학적으로 말이 안 된다(PR 중앙 {pr_med:.1f}ms)")

# ── 리듬 특징 — ★ `rhythm` 팔의 **입력**이자 다른 팔의 **잔차화·매칭 축**이다
_S = pd.Series(pre); _G = _S.groupby(pd.Series(RID))
def local_base(k):
    r = np.asarray(_G.apply(lambda x: x.shift(1).rolling(k, min_periods=1).median())).astype(float)
    return np.where(np.isfinite(r), r, pre)
_med = _G.transform("median").to_numpy()
_std = _G.transform("std").to_numpy(); _mean = _G.transform("mean").to_numpy()
f1 = _med - pre
f2 = {k: 1.0 - pre / (local_base(k) + 1e-9) for k in FULL_K}
f3 = post - pre
f4 = np.nan_to_num(_std / (_mean + 1e-9))
RHY = np.nan_to_num(np.c_[f1, np.column_stack([f2[k] for k in RHY_K]), f3, f4,
                          np.log1p(np.clip(pre, 0, None)), np.log1p(np.clip(post, 0, None))],
                    nan=0.0, posinf=0.0, neginf=0.0)
run.log(f"  리듬 특징 {RHY.shape[1]}차원 — `rhythm` 팔의 입력이자 잔차화 축")

def basis_ext(idx):
    return np.c_[np.ones(len(idx)), f1[idx], f3[idx], f4[idx],
                 np.column_stack([f2[k][idx] for k in FULL_K])]

def resid(v, idx):
    X = basis_ext(idx); y = np.asarray(v, float)
    okm = np.isfinite(y)
    if okm.sum() < X.shape[1] + 5:
        return np.full(len(idx), np.nan)
    b = np.linalg.lstsq(X[okm], y[okm], rcond=None)[0]
    return y - X @ b

def matched_auc(vsub, idx, w=None):
    """★★ 층 폭 `w` 샘플로 f1 을 묶는다. 1차는 `np.round(f1)` = **1샘플** 이었다."""
    w = STRATUM_W if w is None else w
    tt = TT[idx]; key = np.round(np.asarray(f1[idx], float) / float(w)).astype(int)
    win = tie = tot = 0.0
    for kk in np.unique(key):
        m = np.where(key == kk)[0]
        a = vsub[m[tt[m]]]; b = vsub[m[~tt[m]]]
        a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
        if not len(a) or not len(b):
            continue
        d = a[:, None] - b[None, :]
        win += float((d > 0).sum()); tie += float((d == 0).sum()); tot += float(d.size)
    return ((win + 0.5 * tie) / tot, int(tot)) if tot >= MIN_PAIR else (float("nan"), int(tot))

def pair_count(idx, w):
    tt = TT[idx]; key = np.round(np.asarray(f1[idx], float) / float(w)).astype(int)
    tot = 0
    for kk in np.unique(key):
        m = key == kk
        tot += int(tt[m].sum()) * int((~tt[m]).sum())
    return tot

# ── 시간 분할
t_sec = rsmp / FS
TRN, EVL = {}, {}
rows = []
for r in RS:
    ii = np.where(RID == r)[0]
    t = t_sec[ii]; t0, t1 = t.min(), t.max()
    cut = t0 + FRAC_TRAIN * (t1 - t0)
    tr = ii[t < cut - GUARD_S / 2.0]; ev = ii[t > cut + GUARD_S / 2.0]
    TRN[int(r)], EVL[int(r)] = tr, ev
    rows.append(dict(rec=int(r), tr_s=int(TT[tr].sum()), tr_n=int((~TT[tr]).sum()),
                     ev_s=int(TT[ev].sum()), ev_n=int((~TT[ev]).sum()),
                     pairs=int(pair_count(ev, STRATUM_W)) if len(ev) else 0,
                     burden=float(TT[ii].mean())))

# ── ★★★ 코호트 = **매칭 쌍** 기준 (1차는 S/N 개수로 세서 35→11 로 죽었다)
OKR = [d for d in rows if d["tr_s"] >= MIN_S_TR and d["tr_n"] >= MIN_N_TR
       and d["pairs"] >= MIN_PAIR]
COH = [d["rec"] for d in OKR]
old_rule = [d for d in rows if d["tr_s"] >= 25 and d["tr_n"] >= 25
            and d["ev_s"] >= 25 and d["ev_n"] >= 25]
run.log(f"\n  ★★ 코호트 기준 비교 — 1차(S/N 개수) **{len(old_rule)}명** vs "
        f"이번(매칭 쌍 ≥{MIN_PAIR}) **{len(COH)}명**")
run.log(f"     1차는 {len(old_rule)}명을 코호트로 잡았는데 통계량은 **{REF['g1_n']}명**에서만 "
        "섰다 — 세기와 요구가 달랐다")
pv = np.array([d["pairs"] for d in rows], float)
run.log(f"  평가 구간 매칭 쌍 — 중앙 {np.median(pv):.0f} · 사분위 "
        f"[{np.percentile(pv,25):.0f}, {np.percentile(pv,75):.0f}] · 최대 {pv.max():.0f}")
run.log(f"  {'rec':>5}{'학습S':>7}{'학습N':>7}{'평가S':>7}{'평가N':>7}{'쌍':>10}{'부담':>8}")
for d in sorted(OKR, key=lambda x: -x["burden"])[:10]:
    run.log(f"  {d['rec']:>5}{d['tr_s']:>7}{d['tr_n']:>7}{d['ev_s']:>7}{d['ev_n']:>7}"
            f"{d['pairs']:>10,}{d['burden']:>8.4f}")
if len(COH) < MIN_REC:
    raise AssetError(f"코호트가 {len(COH)}명뿐이다(<{MIN_REC}) — 세기 결과만 남기고 **종결**")
g_("H0", "✅ 지지", f"정합·생리 타당성 통과 · **매칭 쌍 기준** 코호트 {len(COH)}명")
CONFIG["H0"] = dict(n_rec=int(len(RS)), cohort=COH, n_cohort=len(COH),
                    n_old_rule=len(old_rule), rows=rows, pr_med=pr_med)
run.save_json("config", CONFIG)


In [ ]:
# CELL 3 — 【H-1】 ★ 자 세우기 — 층 폭 민감도 · **런 내 천장 재계산**
run.log("\n" + "=" * 100)
run.log("【H-1】 자 세우기 — 층 폭 민감도 · 런 내 천장 (관문 아님 · R35 ①)")
run.log("=" * 100)

def stat_of(rec, score_ev, w=None):
    """★★★ 천장과 **같은 통계량** — 리듬 잔차화 후 f1 층 매칭."""
    ev = EVL[rec]
    return matched_auc(resid(score_ev, ev), ev, w=w)

run.log(f"  {'층 폭':>8}{'ms':>8}{'채점가능':>10}{'중앙 쌍':>12}{'p_score 천장':>14}")
GRID_OUT = {}
for w in W_GRID:
    okc, per, prs = 0, [], []
    for rec in COH:
        ev = EVL[rec]
        prs.append(pair_count(ev, w))
        a, _ = matched_auc(resid(psc_asset[ev], ev), ev, w=w)
        if np.isfinite(a):
            okc += 1; per.append(a)
    m_ = float(np.mean(per)) if per else float("nan")
    GRID_OUT[w] = dict(n_ok=okc, med_pairs=float(np.median(prs)), ceiling=m_)
    star = "  ← **판정 폭(사전등록)**" if w == STRATUM_W else ""
    run.log(f"  {w:>6}샘플{w/FS*1000:>8.1f}{okc:>10}{np.median(prs):>12,.0f}{m_:>14.4f}{star}")
run.log("  ▸ ★ 격자는 **보고용**이다 — 판정은 사전등록 폭 하나로만 한다(문턱 훑기 금지 · R34 ②)")

base = [stat_of(rec, psc_asset[EVL[rec]])[0] for rec in COH]
bm, blo, bhi, bn = boot_mean(base, SEED0 + 21, NB_BOOT)
run.log(f"\n  ★★ **런 내 천장** — 평가 구간 `p_score` 매칭 AUROC **{bm:.4f}** "
        f"[{blo:.4f}, {bhi:.4f}] · n={bn}")
run.log(f"     ⚠️ 1차 앵커 {ANCHOR_OLD} 는 **층 폭 1샘플·다른 모집단**의 수라 **비교 불가**다.")
run.log(f"        (참고 1차 런 내 기준선 {REF['g1_base']} · Q7-AA AA1 {REF['aa1']})")
run.log("     ▸ H2 의 두 번째 조건은 **이 천장의 CI 상단**을 쓴다 — 자가 바뀌면 눈금도 다시 긋는다")
g_("H1", "✅ 지지", f"층 폭 {STRATUM_W}샘플에서 채점 가능 {bn}명 · 천장 {bm:.4f} 확보")
CONFIG["H1"] = dict(grid={str(k): v for k, v in GRID_OUT.items()},
                    ceiling=dict(mean=bm, lo=blo, hi=bhi, n=int(bn)))
run.save_json("config", CONFIG)


In [ ]:
# CELL 4 — 【H-A】 5팔 구성 · 고정 모델 (파형 CNN + 리듬 MLP)
import torch
import torch.nn as nn
run.log("\n" + "=" * 100)
run.log("【H-A】 5팔 구성 · 고정 모델")
run.log("=" * 100)
DEV_T = "cuda" if torch.cuda.is_available() else "cpu"
PMASK_HW = int(round(PMASK_MS * FS / 1000.0))
W = 2 * HW_P + 1
run.log(f"  torch {torch.__version__} · device {DEV_T}")
run.log(f"  P 창 {W}샘플({W/FS*1000:.0f}ms) · P 마스크 ±{PMASK_HW}샘플({PMASK_MS:.0f}ms · **P 전폭**)")
run.log(f"  대조창 `offwin` 중심 {OFFWIN_C} (R+{OFFWIN_C-RPRE}샘플 ≈ TP 구간 · P 창과 비겹침)")

def anchors(rec):
    ii = np.concatenate([TRN[rec], EVL[rec]])
    a = pidx0[ii].astype(float); good = a[a >= 0]
    fill = float(np.median(good)) if len(good) else float(RPRE - 0.16 * FS)
    return {int(k): (int(a[j]) if a[j] >= 0 else int(round(fill))) for j, k in enumerate(ii)}

def windows(idx, arm, anc, template=None):
    """★ 파형 팔은 **같은 창 위치**를 본다 — `offwin` 만 의도적으로 다른 자리다."""
    X = XB[idx]
    if arm in ("cancel", "pmask"):
        X = X - template[None, :]
    out = np.zeros((len(idx), W), float)
    for j, gi in enumerate(idx):
        c = OFFWIN_C if arm == "offwin" else anc[int(gi)]
        lo, hi = c - HW_P, c + HW_P + 1
        a, b = max(lo, 0), min(hi, X.shape[1])
        out[j, a - lo:b - lo] = X[j, a:b]
    if arm == "pmask":
        out[:, max(HW_P - PMASK_HW, 0):HW_P + PMASK_HW + 1] = 0.0
    return out

class TinyCNN(nn.Module):
    """파형 팔 — 약 740 파라미터."""
    def __init__(self):
        super().__init__()
        self.c1 = nn.Conv1d(1, 8, 7, padding=3)
        self.c2 = nn.Conv1d(8, 16, 5, padding=2)
        self.fc = nn.Linear(16, 1)
    def forward(self, x):
        h = torch.relu(self.c1(x)); h = torch.max_pool1d(h, 2)
        h = torch.relu(self.c2(h)); h = torch.mean(h, dim=2)
        return self.fc(h).squeeze(1)

class TinyMLP(nn.Module):
    """★ `rhythm` 팔 — 입력 9차원이라 은닉 48 이면 **용량 부족이 아니다**(약 530 파라미터)."""
    def __init__(self, d):
        super().__init__()
        self.h1 = nn.Linear(d, 48); self.h2 = nn.Linear(48, 1)
    def forward(self, x):
        return self.h2(torch.relu(self.h1(x))).squeeze(1)

def fit_predict(Xtr, ytr, Xev, seed, kind):
    """표준화는 **학습 구간 통계**로만(R22)."""
    torch.manual_seed(seed); np.random.seed(seed % (2**31))
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-9
    xt = torch.tensor((Xtr - mu) / sd, dtype=torch.float32, device=DEV_T)
    xe = torch.tensor((Xev - mu) / sd, dtype=torch.float32, device=DEV_T)
    if kind == "wave":
        xt = xt.unsqueeze(1); xe = xe.unsqueeze(1); m = TinyCNN().to(DEV_T)
    else:
        m = TinyMLP(Xtr.shape[1]).to(DEV_T)
    yt = torch.tensor(ytr, dtype=torch.float32, device=DEV_T)
    npos, nneg = float(ytr.sum()), float((1 - ytr).sum())
    pw = torch.tensor(max(nneg, 1.0) / max(npos, 1.0), dtype=torch.float32, device=DEV_T)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    lossf = nn.BCEWithLogitsLoss(pos_weight=pw)
    m.train()
    for _ in range(EPOCHS):
        opt.zero_grad(); lossf(m(xt), yt).backward(); opt.step()
    m.eval()
    with torch.no_grad():
        return m(xe).cpu().numpy().astype(float)

def run_patient(rec, arm, perm_train=None, shuf_anchor=None, seed=0):
    tr, ev = TRN[rec], EVL[rec]
    if arm == "rhythm":
        Xtr, Xev, kind = RHY[tr], RHY[ev], "vec"
    else:
        anc = anchors(rec)
        if shuf_anchor is not None:
            ks = list(anc.keys()); vs = [anc[k] for k in ks]
            vs = [vs[i] for i in shuf_anchor.permutation(len(vs))]
            anc = dict(zip(ks, vs))
        tmpl = np.median(XB[tr], axis=0) if arm in ("cancel", "pmask") else None
        Xtr, Xev, kind = windows(tr, arm, anc, tmpl), windows(ev, arm, anc, tmpl), "wave"
    ytr = TT[tr].astype(float)
    if perm_train is not None:
        ytr = ytr[perm_train.permutation(len(ytr))]
    sc = fit_predict(Xtr, ytr, Xev, seed, kind)
    return stat_of(rec, sc)

run.log("  ▸ `rhythm` 팔도 **같은 잔차화·매칭**을 거친다 → 거기 남는 건 **비선형 리듬**이고,")
run.log("    그게 1차에서 의심한 교란(정렬 비의존 신호)의 크기다")


In [ ]:
# CELL 5 — 【H-B】 ★★★ H2 — 주 관문: `raw − rhythm`
run.log("\n" + "=" * 100)
run.log("【H-B】 H2 — **주 관문**: P 창이 리듬 너머로 더하는가")
run.log("=" * 100)
T0 = time.time()
H2 = {}
for arm in ARMS:
    per = [run_patient(rec, arm, seed=SEED0 + 1000 + rec)[0] for rec in COH]
    m_, lo_, hi_, n_ = boot_mean(per, SEED0 + 31, NB_BOOT)
    H2[arm] = dict(per={int(r): float(v) for r, v in zip(COH, per)},
                   mean=m_, lo=lo_, hi=hi_, n=int(n_), mde=float(mde(lo_, hi_)))
    run.log(f"  {arm:<10} 매칭 AUROC **{m_:.4f}** [{lo_:.4f}, {hi_:.4f}] · n={n_} · "
            f"MDE {mde(lo_, hi_):.4f}")
CEILM = CONFIG["H1"]["ceiling"]["mean"]; CEIL_HI = CONFIG["H1"]["ceiling"]["hi"]
run.log(f"  ({time.time()-T0:.0f}초) · 런 내 천장 {CEILM:.4f} "
        f"[{CONFIG['H1']['ceiling']['lo']:.4f}, {CEIL_HI:.4f}]")

run.log("\n  ★★★ **리듬 대비**(짝지은 차 · 같은 환자) — 이게 **각 창의 순수 몫**이다")
VS_RHY = {}
for arm in WAVE_ARMS:
    va = [H2["rhythm"]["per"][r] for r in COH]
    vb = [H2[arm]["per"][r] for r in COH]
    pm, plo, phi, pn = boot_pair(va, vb, SEED0 + 41, NB_BOOT)
    VS_RHY[arm] = dict(mean=pm, lo=plo, hi=phi, n=int(pn), mde=float(mde(plo, phi)))
    run.log(f"    {arm:<10} − rhythm  **{pm:+.4f}** [{plo:+.4f}, {phi:+.4f}] · n={pn}")
run.log(f"    ▸ 성분 — rhythm {H2['rhythm']['mean']:.4f} · raw {H2['raw']['mean']:.4f} "
        "(차의 부호를 성분 없이 인용하지 않는다 · R36 ⑤)")

d_raw = VS_RHY["raw"]
c2_ = bool(H2["raw"]["lo"] > CEIL_HI)
run.log(f"\n  ★★ H2 조건 ⓑ — `raw` 가 천장 CI 상단({CEIL_HI:.4f}) 초과? "
        f"{'예' if c2_ else '**아니오**'} (raw CI 하단 {H2['raw']['lo']:.4f})")
run.log("  ▸ ★ 조건 ⓐ(`raw − rhythm`)의 판정은 **H3 이 대비의 영점을 잰 뒤**에 한다 —")
run.log("    이 대비의 영점은 **0 이 아니다**(`rhythm` 팔이 잔차화로 0.5 아래에 앉는다)")
CONFIG["H2"] = {k: {kk: vv for kk, vv in v.items() if kk != "per"} for k, v in H2.items()}
CONFIG["H2_per"] = {k: v["per"] for k, v in H2.items()}
CONFIG["H2_vs_rhythm"] = VS_RHY
CONFIG["H2_cond"] = dict(over_ceiling=c2_, ceil_hi=float(CEIL_HI))
run.save_json("config", CONFIG)


In [ ]:
# CELL 6 — 【H-C】 H3 영점 · ★★ H4 정렬·위치 대조
run.log("\n" + "=" * 100)
run.log("【H-C】 H3 — 파이프라인 영점 · H4 — 정렬·위치 대조")
run.log("=" * 100)
T1 = time.time()
# ★★★ 영점을 **(rep, 환자)별로 보관**한다 — 그래야 **대비의 영점**을 만들 수 있다.
#     1판 스모크(음성 조건)가 잡은 결함: `rhythm` 팔은 자기 점수가 리듬 기저에 잔차화되며
#     계통적으로 0.5 **아래**에 앉는다(합성 실측 0.4134). 그래서 신호가 0 인데도
#     `raw − rhythm` 이 **+0.0907** 이 나왔다. 즉 **이 대비의 영점은 0 이 아니다.**
NUL_PER = {a: {} for a in ARMS}
H3 = {}
for arm in ARMS:
    for rep in range(N_PERM_H3):
        for rec in COH:
            pr = np.random.RandomState(SEED0 + 5000 + 97 * rep + rec)
            NUL_PER[arm][(rep, rec)] = run_patient(
                rec, arm, perm_train=pr, seed=SEED0 + 6000 + 97 * rep + rec)[0]
    vals = list(NUL_PER[arm].values())
    m_, lo_, hi_, n_ = boot_mean(vals, SEED0 + 51, NB_BOOT)
    H3[arm] = dict(mean=m_, lo=lo_, hi=hi_, n=int(n_))
    run.log(f"  H3 {arm:<10} 영점 **{m_:.4f}** [{lo_:.4f}, {hi_:.4f}] · n={n_}")
run.log(f"  ({time.time()-T1:.0f}초 · reps={N_PERM_H3})")

# ── ★★★ **대비의 영점** — 같은 라벨 치환 하에서 같은 짝지은 차를 만든다
run.log("\n  ★★★ **대비의 영점**(라벨 치환 하 같은 짝지은 차) — 0 을 가정하지 않는다")
NUL_DIFF = {}
keys = sorted(NUL_PER["rhythm"].keys())
for arm in WAVE_ARMS:
    dv = [NUL_PER[arm][k] - NUL_PER["rhythm"][k] for k in keys]
    m_, lo_, hi_, n_ = boot_mean(dv, SEED0 + 52, NB_BOOT)
    NUL_DIFF[arm] = dict(mean=m_, lo=lo_, hi=hi_, n=int(n_))
    run.log(f"    {arm:<10} − rhythm 영점 **{m_:+.4f}** [{lo_:+.4f}, {hi_:+.4f}] · n={n_}")
run.log(f"    ▸ `rhythm` 영점 {H3['rhythm']['mean']:.4f} 이 0.5 에서 벗어나면 이 대비의 "
        "영점도 0 이 아니다 — **관측 대비는 이 값 기준으로 읽는다**")
g_("H3", "✅ 지지" if all(np.isfinite(H3[a]["mean"]) for a in ARMS) else "⚠️ 미결",
   f"팔별 영점과 **대비의 영점**을 둘 다 측정했다 (`raw−rhythm` 영점 "
   f"{NUL_DIFF['raw']['mean']:+.4f})")

# ── ★★★ 이제 H2 를 판정한다 — 대비의 **측정된 영점** 기준
NR = NUL_DIFF["raw"]
c1_ = decide(d_raw["lo"], d_raw["hi"], NR["hi"], ">")
run.log(f"\n  ★★★ **H2 판정** — `raw − rhythm` {d_raw['mean']:+.4f} "
        f"[{d_raw['lo']:+.4f}, {d_raw['hi']:+.4f}] vs **영점 상단 {NR['hi']:+.4f}**")
run.log(f"     ⓐ {c1_}  ·  ⓑ 천장 초과 {'예' if c2_ else '**아니오**'}")
if c1_.startswith("✅") and c2_:
    g_("H2", "✅ 지지",
       f"P 창이 리듬 너머로 **영점 초과분 {d_raw['mean']-NR['mean']:+.4f}** 만큼 더하고 천장도 넘는다")
elif c1_.startswith("❌"):
    g_("H2", "❌ 기각",
       f"`raw − rhythm` 이 **자기 영점({NR['mean']:+.4f})을 못 넘는다** — P 창이 리듬 너머로 안 더한다")
else:
    which = "ⓐ 차(영점 대비)" if not c1_.startswith("✅") else "ⓑ 천장"
    g_("H2", "⚠️ 미결",
       f"두 조건 중 **{which}** 를 못 넘었다 — **등가가 아니다**(상한 {d_raw['hi']:+.4f})")
CONFIG["H2_cond"] = dict(diff_verdict=c1_, over_ceiling=c2_, ceil_hi=float(CEIL_HI),
                         null_diff={k: v for k, v in NUL_DIFF.items()})
run.save_json("config", CONFIG)

# ── H4ⓑ `p_idx` 단독(학습 없음) · 앵커 산포
g4a, disp = [], []
for rec in COH:
    ev = EVL[rec]; anc = anchors(rec)
    v = np.array([anc[int(g)] for g in ev], float)
    a, _ = matched_auc(resid(v, ev), ev)
    g4a.append(a)
    allv = np.array([anc[int(g)] for g in np.concatenate([TRN[rec], EVL[rec]])], float)
    disp.append(float(allv.std()))
am, alo, ahi, an = boot_mean(g4a, SEED0 + 71, NB_BOOT)
DISP = float(np.median(disp))
run.log(f"\n  H4ⓑ `p_idx` 단독 매칭 AUROC **{am:.4f}** [{alo:.4f}, {ahi:.4f}] · n={an}")
run.log(f"       앵커 산포(중앙 SD) {DISP:.1f}샘플 = 창 반폭 {HW_P} 의 {DISP/HW_P:.0%}")

# ── H4ⓐ 앵커 셔플 (offwin 은 앵커와 무관하므로 정의되지 않는다)
T2_ = time.time()
H4 = {}
for arm in ("raw", "cancel", "pmask"):
    sh = []
    for rep in range(N_SHUF_H4):
        for rec in COH:
            sr = np.random.RandomState(SEED0 + 7000 + 97 * rep + rec)
            sh.append(run_patient(rec, arm, shuf_anchor=sr,
                                  seed=SEED0 + 8000 + 97 * rep + rec)[0])
    m_, lo_, hi_, n_ = boot_mean(sh, SEED0 + 61, NB_BOOT)
    H4[arm] = dict(mean=m_, lo=lo_, hi=hi_, n=int(n_))
    run.log(f"  H4ⓐ {arm:<10} 셔플 **{m_:.4f}** [{lo_:.4f}, {hi_:.4f}] · "
            f"관측 {H2[arm]['mean']:.4f} · 영점 {H3[arm]['mean']:.4f}")
run.log(f"  ({time.time()-T2_:.0f}초 · reps={N_SHUF_H4})")

# ── ★★ H4ⓒ 위치 대조 — `offwin` 이 `raw` 와 같으면 P 창이 특별하지 않다
po, plo_, phi_, pn_ = boot_pair([H2["offwin"]["per"][r] for r in COH],
                                [H2["raw"]["per"][r] for r in COH], SEED0 + 81, NB_BOOT)
run.log(f"\n  ★★ H4ⓒ **위치 대조** — `raw − offwin` **{po:+.4f}** [{plo_:+.4f}, {phi_:+.4f}]")
run.log(f"       성분 — raw−rhythm {VS_RHY['raw']['mean']:+.4f} · "
        f"offwin−rhythm {VS_RHY['offwin']['mean']:+.4f}")
OFF_SAME = not (plo_ > 0)
if OFF_SAME:
    run.log("       ⚠️⚠️ **P 창이 창 밖보다 낫다는 증거가 없다** — 신호가 위치와 무관하다는 뜻이고,")
    run.log("          그러면 H2 를 P 형태의 증거로 읽지 않는다(리듬 대리변수가 유력하다)")
g_("H4", "✅ 지지" if not OFF_SAME else "❌ 기각",
   f"`raw` 가 `offwin` 을 **{po:+.4f}** 로 앞선다 — P 창이 특별하다" if not OFF_SAME else
   "★★ **P 창이 창 밖과 구별되지 않는다** — 위치 비의존 신호이므로 P 형태의 증거가 아니다")
CONFIG["H3"] = H3; CONFIG["H3_null_diff"] = NUL_DIFF
CONFIG["H4"] = dict(shuffle=H4, pidx_alone=dict(mean=am, lo=alo, hi=ahi, n=int(an)),
                    disp=DISP, disp_frac=float(DISP / HW_P),
                    raw_minus_offwin=dict(mean=po, lo=plo_, hi=phi_, n=int(pn_)),
                    off_same=bool(OFF_SAME))
run.save_json("config", CONFIG)


In [ ]:
# CELL 7 — 【H-D】 필요표본 · ★ H5 검산표 · 그림 · 요약
run.log("\n" + "=" * 100)
run.log("【H-D】 필요표본 · H5 결론 검산표")
run.log("=" * 100)
run.log(f"  필요표본 (**우월 프레임** · 단위 = 환자 · 현재 {len(COH)}명 · 대비 = rhythm)")
NEED = {}
for arm in WAVE_ARMS:
    d = VS_RHY[arm]; eff, half = d["mean"], d["mde"]
    n5 = need_super(len(COH), half, eff, False); n8 = need_super(len(COH), half, eff, True)
    zero = abs(eff) < half
    NEED[arm] = dict(effect=float(eff), half=float(half), sup50=float(n5), sup80=float(n8),
                     uninterpretable=bool(zero))
    run.log(f"  {arm:<10}{eff:>+9.4f}{half:>9.4f}{n5:>9.0f}{n8:>9.0f}  "
            + ("★ **효과 ≈ 0 이라 해석 불가**(R41 ②)" if zero else "읽을 수 있다"))
run.log("    ▸ 판정은 필요표본이 아니라 **MDE 로** 한다")

run.log("\n  ★ H5 — **결론 검산표**")
CHECK = [
    dict(claim=f"자를 다시 세웠다 — 층 폭 {STRATUM_W}샘플({STRATUM_W/FS*1000:.1f}ms) · "
               f"런 내 천장 {CEILM:.4f}",
         num=f"코호트 {len(COH)}명(매칭 쌍 ≥{MIN_PAIR}) · 1차 기준이면 "
             f"{CONFIG['H0']['n_old_rule']}명이었지만 1차 통계량은 {REF['g1_n']}명에서만 섰다",
         assume="층 폭을 넓혀도 **심박 통제가 유지**된다는 것(27.8ms 안이면 같은 심박으로 본다)",
         iffalse=f"폭이 너무 넓으면 리듬이 새 들어온다 — 그래서 격자 {W_GRID} 를 함께 찍었다"),
    dict(claim=f"H2 주 관문 `raw − rhythm` {VS_RHY['raw']['mean']:+.4f} "
               f"[{VS_RHY['raw']['lo']:+.4f}, {VS_RHY['raw']['hi']:+.4f}]",
         num=f"성분 rhythm {H2['rhythm']['mean']:.4f} · raw {H2['raw']['mean']:.4f} · "
             f"MDE {VS_RHY['raw']['mde']:.4f}",
         assume="`rhythm` 팔이 **비선형 리듬을 충분히 흡수**한다는 것(입력 9차원 · 은닉 48)",
         iffalse="리듬 팔이 약하면 `raw − rhythm` 이 **과대평가**된다 — 용량이 아니라 특징 집합의 "
                 "한계일 수 있다(RR 기반 9차원 밖의 리듬 정보는 못 담는다)"),
    dict(claim=f"영점을 측정했다 — 팔별 **그리고 대비별** (reps={N_PERM_H3})",
         num="팔별 " + " · ".join(f"{a} {H3[a]['mean']:.4f}" for a in ARMS)
             + f" | `raw−rhythm` 영점 {NUL_DIFF['raw']['mean']:+.4f} "
               f"[{NUL_DIFF['raw']['lo']:+.4f}, {NUL_DIFF['raw']['hi']:+.4f}]",
         assume="**없음** — 학습 라벨만 치환하고 같은 절차를 재학습한다",
         iffalse="★★ `rhythm` 팔은 잔차화로 0.5 **아래**에 앉으므로 `raw−rhythm` 의 영점은 "
                 "**0 이 아니다**. 0 으로 판정하면 신호 0 에서도 통과한다(스모크 음성 조건 실측 "
                 "+0.0907) — 그래서 **측정된 영점**으로 판정한다"),
    dict(claim=("H4ⓒ P 창이 창 밖보다 낫다" if not OFF_SAME else
                "★★ H4ⓒ **P 창이 창 밖과 구별되지 않는다**"),
         num=f"`raw − offwin` {po:+.4f} [{plo_:+.4f}, {phi_:+.4f}]",
         assume=f"`offwin`(중심 {OFFWIN_C})이 **P 를 안 담는다**는 것",
         iffalse="대조창이 P 를 조금이라도 담으면 이 차는 **하한**이다"),
    dict(claim=f"앵커 셔플 대조의 검정력 — 산포 {DISP:.1f} vs 창 반폭 {HW_P}({DISP/HW_P:.0%})",
         num=" · ".join(f"{a} {H4[a]['mean']:.4f}" for a in H4),
         assume="셔플이 창 내용을 실제로 바꾼다는 것",
         iffalse="산포 ≪ 창이면 이 대조엔 검정력이 없다 — 그래서 **`offwin`(위치 대조)** 을 "
                 "따로 뒀고, 위치 질문은 그쪽으로 답한다"),
    dict(claim="1차의 수를 인용하지 않는다",
         num=f"1차 raw {REF['g1_raw']} 는 층 폭 1샘플·n={REF['g1_n']} 의 수다",
         assume="**없음** — 자가 달라 비교 불가다",
         iffalse="—"),
    dict(claim="누출·선택 편의가 없다",
         num=f"시간 분할 앞{FRAC_TRAIN:.0%}/뒤 · 가드밴드 {GUARD_S:.0f}초 · 템플릿·표준화 모두 "
             f"학습 구간 · 고정 구조 · EPOCHS={EPOCHS} · 용량 격자 없음",
         assume="**없음** — 구성으로 보장된다",
         iffalse="—"),
]
for i, c in enumerate(CHECK, 1):
    run.log(f"\n  [{i}] **{c['claim']}**")
    run.log(f"      근거   {c['num']}")
    run.log(f"      가정   {c['assume']}")
    run.log(f"      틀리면 {c['iffalse']}")
CONFIG["need"] = NEED; CONFIG["H5"] = CHECK
run.save_json("config", CONFIG)

# ── 그림 (축·범례는 ASCII 만)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.6))
xs = np.arange(len(ARMS))
obs = [H2[a]["mean"] for a in ARMS]
oe = [[obs[i] - H2[a]["lo"] for i, a in enumerate(ARMS)],
      [H2[a]["hi"] - obs[i] for i, a in enumerate(ARMS)]]
nul = [H3[a]["mean"] for a in ARMS]
ax[0].errorbar(xs - 0.12, obs, yerr=oe, fmt="o", capsize=5, color="tab:red", label="observed")
ax[0].scatter(xs + 0.12, nul, marker="x", s=60, color="tab:gray", label="pipeline null")
ax[0].axhline(CEILM, ls="--", color="tab:blue", lw=1.2, label=f"in-run ceiling {CEILM:.3f}")
ax[0].axhline(0.5, color="k", lw=.8)
ax[0].set_xticks(xs); ax[0].set_xticklabels(list(ARMS), fontsize=8, rotation=15)
ax[0].set_ylabel("matched AUROC (rhythm-residualised)")
ax[0].set_title(f"H2 : arms (stratum {STRATUM_W} samp, n={len(COH)})", fontsize=9)
ax[0].legend(fontsize=6); ax[0].grid(alpha=.3, axis="y")

nm = list(WAVE_ARMS)
vv = [VS_RHY[a]["mean"] for a in nm]
lo2 = [vv[i] - VS_RHY[a]["lo"] for i, a in enumerate(nm)]
hi2 = [VS_RHY[a]["hi"] - vv[i] for i, a in enumerate(nm)]
ax[1].errorbar(vv, np.arange(len(nm)), xerr=[lo2, hi2], fmt="o", capsize=5, color="tab:red")
ax[1].axvline(0, color="k", lw=.9)
ax[1].set_yticks(range(len(nm))); ax[1].set_yticklabels([f"{a} - rhythm" for a in nm], fontsize=8)
ax[1].set_xlabel("paired difference vs rhythm-only arm")
ax[1].set_title("H2 : pure contribution of each window", fontsize=9)
ax[1].grid(alpha=.3, axis="x")

ws = list(W_GRID)
ax[2].plot(ws, [GRID_OUT[w]["n_ok"] for w in ws], "o-", color="tab:green")
ax[2].axvline(STRATUM_W, ls="--", color="tab:blue", lw=1.2)
ax[2].set_xscale("log"); ax[2].set_xlabel("stratum width (samples)")
ax[2].set_ylabel("scorable patients", color="tab:green")
ax2b = ax[2].twinx()
ax2b.plot(ws, [GRID_OUT[w]["med_pairs"] for w in ws], "s--", color="tab:orange")
ax2b.set_yscale("log"); ax2b.set_ylabel("median matched pairs", color="tab:orange")
ax[2].set_title("H1 : the ruler (1 samp = 2.8 ms killed the pairs)", fontsize=9)
ax[2].grid(alpha=.3)
fig.tight_layout()
PNG = run.save_fig("q9_g1p_pmorph_v2", fig)
plt.close(fig); display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("요약")
run.log("=" * 100)
ok_ = lambda k: VERD.get(k, "").startswith("✅")
for g in READ_ORDER[:5]:
    run.log(f"  {g:<5}{VERD.get(g, '(미실행)')}")
run.log("")
run.log(f"  환자 {len(COH)}명 · 층 폭 {STRATUM_W}샘플 · 런 내 천장 {CEILM:.4f}")
for a in ARMS:
    extra = f" · vs rhythm {VS_RHY[a]['mean']:+.4f}" if a in VS_RHY else ""
    run.log(f"    {a:<10}{H2[a]['mean']:.4f} [{H2[a]['lo']:.4f}, {H2[a]['hi']:.4f}] · "
            f"영점 {H3[a]['mean']:.4f}{extra}")
run.log("")
if OFF_SAME:
    run.log("  ⛔⛔ **H4ⓒ 가 갈래를 닫는다 — P 창이 창 밖과 구별되지 않는다.**")
    run.log(f"     `raw − offwin` {po:+.4f} [{plo_:+.4f}, {phi_:+.4f}] — 신호가 **위치와 무관**하다.")
    run.log("     그러면 H2 가 무엇이든 **P 형태의 증거가 아니다**. 층① 을 종결한다")
    run.log("     → 층②(Q4 `burden-feature`)로 간다")
elif ok_("H2"):
    run.log(f"  ★★★ **H2 통과 — P 창이 리듬 너머로 {VS_RHY['raw']['mean']:+.4f} 더한다.**")
    run.log(f"     그리고 학습 표현이 런 내 천장({CEILM:.4f})도 넘었다 → **층① 열림**")
    run.log("     → 다음은 **G2 라벨-이득 곡선** (임상 결정 숫자는 AUROC 가 아니라 **N**)")
    if VS_RHY["cancel"]["lo"] > VS_RHY["raw"]["hi"]:
        run.log("     ★ 그리고 **소거가 원신호를 앞선다** — 판별력에서 소거가 처음 살아났다")
else:
    run.log("  ⛔ **H2 미달 — P 창이 리듬 너머로 더한다는 증거가 없다.**")
    run.log(f"     `raw − rhythm` {VS_RHY['raw']['mean']:+.4f} "
            f"[{VS_RHY['raw']['lo']:+.4f}, {VS_RHY['raw']['hi']:+.4f}] · "
            f"MDE {VS_RHY['raw']['mde']:.4f}")
    run.log("     최종 문장:")
    run.log("       「2리드 홀터 SVEB 검출에서, **환자 자신의 라벨로 학습**하고 리듬-only 개인화")
    run.log("        모델을 기저로 깔면, P 창 표현의 증분은 위 CI 상단 이하다. Q7-AA 의 해리는")
    run.log("        **「환자 안에서도 리듬 너머로는 아니다」**로 좁혀진다.」")
    run.log("     → 층① 종결. 층②(Q4 `burden-feature`)로 간다")
run.log("")
run.log("  ▸ ★ 1차의 수(raw 0.6475 등)는 **자가 달라 인용하지 않는다**")
run.log("  ▸ ★ 미결은 **등가가 아니다** — 상한은 CI 상단이다(R33 ① · R36 ①)")

run.finish({
    "exp_id": "quest46_q9_g1p_pmorph_v2",
    "metric": "raw_minus_rhythm",
    "value": float(VS_RHY["raw"]["mean"]),
    "passed": bool(ok_("H0") and ok_("H2") and not OFF_SAME),
    "summary": ("G1 재설계 — 층 폭(2.8ms→27.8ms)·매칭 쌍 기준 코호트·런 내 천장으로 자를 고치고, "
                "리듬-only 팔(사전등록 G3 흡수)을 넣어 `raw − rhythm` 을 주 관문으로 삼는다. "
                "위치 대조 `offwin` 과 P 전폭 마스크로 음성 대조를 강화했다."),
    "verdicts": VERD, "diffs": DIFF, "rule_check": RULE_CHECK,
    "H0": CONFIG.get("H0", {}), "H1": CONFIG.get("H1", {}), "H2": CONFIG.get("H2", {}),
    "H2_per": CONFIG.get("H2_per", {}), "H2_vs_rhythm": CONFIG.get("H2_vs_rhythm", {}),
    "H2_cond": CONFIG.get("H2_cond", {}), "H3": CONFIG.get("H3", {}),
    "H3_null_diff": CONFIG.get("H3_null_diff", {}),
    "H4": CONFIG.get("H4", {}), "H5": CONFIG.get("H5", []),
    "need": CONFIG.get("need", {}), "cohort": COH, "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `python pipelines/ingest_run.py --results result.json "
        "--notebook notebooks/quest46_q9_g1p_pmorph_v2.ipynb`")
